# Data Visualization Project Phase 2: Interactive & Business Intelligence Dashboards

---

## Phase Overview

This notebook covers **Phase 2** of the course project, which builds upon the static visualizations created in Phase 1 using Matplotlib and Seaborn. In this phase, we transition to fully **interactive** visualizations using **Plotly** and **Dash**, and develop a **Power BI** business intelligence dashboard.

The phase is divided into the following tasks:

| Task | Description |
|------|-------------|
| **Task 1** | Convert 3 Phase 1 charts into interactive Plotly charts |
| **Task 2** | Build and deploy a multi-view interactive Dash application |
| **Task 3** | Power BI Report with 8+ visuals and DAX measures |
| **Task 4** | Interactive Power BI Dashboard with cross-filtering and slicers |
| **Task 5** | Screen-recorded walkthrough (3–5 minutes) |

---

### Dataset
The same **Melbourne Housing Dataset** from Phase 1 is used throughout this phase.  
- **Source:** https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot  
- **Size:** 13,580 rows × 21 columns (6,858 rows after cleaning)

## Task 1: Plotly Interactive Charts

In this task, we convert three static Matplotlib charts from Phase 1 into fully interactive Plotly equivalents. Each chart includes:
- **Hover tooltips** with meaningful data
- **Zoom and pan** capabilities
- **Click-to-filter** interactivity

The three charts selected for conversion are:

1. **Scatter Plot** — Building Area vs. Sale Price (color-coded by property type + trendline)
2. **Line Chart** — Average Price Over Time (multi-trace by property type) ✦ *satisfies multi-trace requirement*
3. **Bar Chart** — Average Sale Price by Property Type (animated by sale year) ✦ *satisfies animated figure requirement*

In [1]:
print(5)

5


###### Install & Import Libraries

In [2]:
import ensurepip
import sys

ensurepip.bootstrap()

!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install pandas
!{sys.executable} -m pip install plotly
!{sys.executable} -m pip install statsmodels


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Colour palette (consistent with Phase 1)
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B',
          '#44BBA4', '#E94F37', '#393E41']

print("Libraries imported successfully.")

Libraries imported successfully.


###### Load Cleaned Dataset from Phase1

In [6]:

df = pd.read_csv('C:\\Users\\User\\Desktop\\DV\\DV_project\\Melbourne_Housing_Cleaned.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Dataset loaded successfully from the cleaned file.")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

Dataset loaded successfully from the cleaned file.
Shape: 6858 rows × 25 columns


---

### Chart 1: Interactive Scatter Plot — Building Area vs. Sale Price

This interactive scatter plot is the Plotly equivalent of **Chart 3 from Phase 1**. It visualises the relationship between a property's building area (m²) and its sale price (AUD Millions), with each point colour-coded by property type.

**Interactivity features:**
- **Hover tooltips** — show Suburb, Property Type, Building Area, Price, and Rooms
- **Click legend** — toggle individual property types on/off
- **Zoom & pan** — drag to zoom into any region, double-click to reset
- **OLS trendline** — overall linear trend overlaid across all property types

In [7]:
# Filter to properties under 600 m² (same as Phase 1)
scatter_df = df[df['BuildingArea'] < 600].copy()

fig_scatter = px.scatter(
    scatter_df,
    x='BuildingArea',
    y='Price_M',
    color='PropertyType',
    trendline='ols',
    trendline_scope='overall',
    trendline_color_override='#C73E1D',
    color_discrete_map={
        'House':     '#2E86AB',
        'Townhouse': '#A23B72',
        'Unit':      '#F18F01'
    },
    opacity=0.5,
    hover_data={
        'Suburb':       True,
        'Rooms':        True,
        'BuildingArea': ':.0f',
        'Price_M':      ':.2f',
        'PropertyType': True
    },
    labels={
        'BuildingArea': 'Building Area (m²)',
        'Price_M':      'Sale Price (AUD Millions)',
        'PropertyType': 'Property Type'
    },
    title='Building Area vs. Sale Price by Property Type<br><sup>Melbourne Housing Dataset — Properties < 600 m²</sup>'
)

fig_scatter.update_layout(
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='#F8F9FA',
    font=dict(family='Arial', size=13),
    title_font_size=16,
    legend_title_text='Property Type',
    hovermode='closest',
    yaxis=dict(tickprefix='$', ticksuffix='M', gridcolor='#E0E0E0'),
    xaxis=dict(gridcolor='#E0E0E0'),
    height=520
)

fig_scatter.show()

---

### Chart 2: Interactive Multi-Trace Line Chart — Average Price Over Time by Property Type

This interactive line chart is an enhanced Plotly version of **Chart 1 from Phase 1**. Instead of a single average line, this chart displays **three separate traces** — one for each property type (House, Townhouse, Unit) — allowing direct comparison of price trends over time.

**Interactivity features:**
- **Hover tooltips** — show exact month, property type, and average price
- **Multi-trace** — three lines plotted simultaneously ✦ *satisfies multi-trace requirement*
- **Click legend** — isolate any single property type
- **Range slider** — scroll through the time axis interactively
- **Zoom & pan** — focus on any time period

In [8]:
# Prepare monthly average price per property type
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

line_df = (
    df.groupby(['YearMonth', 'PropertyType'])['Price_M']
      .mean()
      .reset_index()
      .rename(columns={'Price_M': 'Avg_Price_M'})
)
line_df = line_df.sort_values('YearMonth')

fig_line = px.line(
    line_df,
    x='YearMonth',
    y='Avg_Price_M',
    color='PropertyType',
    markers=True,
    color_discrete_map={
        'House':     '#2E86AB',
        'Townhouse': '#A23B72',
        'Unit':      '#F18F01'
    },
    hover_data={
        'YearMonth':   True,
        'Avg_Price_M': ':.2f',
        'PropertyType': True
    },
    labels={
        'YearMonth':    'Sale Month',
        'Avg_Price_M':  'Average Price (AUD Millions)',
        'PropertyType': 'Property Type'
    },
    title='Average Property Sale Price Over Time by Property Type<br><sup>Melbourne Housing Dataset — Multi-Trace (2016–2017)</sup>'
)

fig_line.update_layout(
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='#F8F9FA',
    font=dict(family='Arial', size=13),
    title_font_size=16,
    hovermode='x unified',
    yaxis=dict(tickprefix='$', ticksuffix='M', gridcolor='#E0E0E0'),
    xaxis=dict(
        tickangle=45,
        gridcolor='#E0E0E0',
        rangeslider=dict(visible=True)
    ),
    height=540
)

fig_line.update_traces(line=dict(width=2.5), marker=dict(size=6))

fig_line.show()

---

### Chart 3: Animated Bar Chart — Average Sale Price by Property Type (Per Year)

This animated bar chart is an enhanced Plotly version of **Chart 2 from Phase 1**. It shows the average sale price per property type, with an **animation frame for each sale year** (2016 and 2017), allowing the viewer to observe how prices shifted between the two years.

**Interactivity features:**
- **Animation** — play button cycles through each sale year ✦ *satisfies animated figure requirement*
- **Hover tooltips** — show property type, year, and exact average price
- **Click to pause/resume** the animation
- **Zoom & pan** — adjust the view at any frame

In [9]:
# Prepare yearly average price per property type
bar_anim_df = (
    df.groupby(['YearMonth', 'PropertyType'])['Price_M']
      .mean().reset_index().rename(columns={'Price_M': 'Avg_Price_M'})
)
bar_anim_df = bar_anim_df.sort_values('YearMonth')  # مهم للترتيب الصحيح


fig_bar = px.bar(
    bar_anim_df,
    x='PropertyType',
    y='Avg_Price_M',
    color='PropertyType',
    animation_frame='YearMonth',
    text='Avg_Price_M',
    color_discrete_map={
        'House':     '#2E86AB',
        'Townhouse': '#A23B72',
        'Unit':      '#F18F01'
    },
    range_y=[0, 1.6],
    hover_data={
        'PropertyType': True,
        'YearMonth':     True,
        'Avg_Price_M':  ':.3f'
    },
    labels={
        'PropertyType': 'Property Type',
        'Avg_Price_M':  'Average Price (AUD Millions)',
        'SaleYear':     'Sale Year'
    },
    title='Average Sale Price by Property Type — Animated by Year<br><sup>Melbourne Housing Dataset (2016 vs 2017)</sup>'
)

fig_bar.update_traces(
    texttemplate='$%{text:.2f}M',
    textposition='outside'
)

fig_bar.update_layout(
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='#F8F9FA',
    font=dict(family='Arial', size=13),
    title_font_size=16,
    showlegend=False,
    yaxis=dict(tickprefix='$', ticksuffix='M', gridcolor='#E0E0E0'),
    xaxis=dict(gridcolor='#E0E0E0'),
    height=500
)

fig_bar.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200
fig_bar.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 600

fig_bar.show()

## Task 2: Dash Application (Deployed Online)

In this task, we build a multi-view interactive Dash application using the Melbourne Housing Dataset. The app contains:
- **2 interactive charts** linked through Dash Callbacks
- **3 filter controls**: Dropdown (property type), Slider (max price), Radio Buttons (scatter opacity)
- **4 KPI cards** that update dynamically with every filter change

The app is deployed online via Render. Live URL: *(add after deployment)*

In [12]:
pip install dash plotly statsmodels

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.2 MB 3.5 MB/s eta 0:00:02
   ----------------- ---------------------- 3.1/7.2 MB 6.7 MB/s eta 0:00:01
   ---------------------------- ----------- 5.2/7.2 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 8.0 MB/s  0:00:01

   ----- ---------------------------------- 1/8 [Werkzeug]
   ----- ---------------------------------- 1/8 [Werkzeug]
   ----- ---------------------------------- 1/8 [Werkzeug]
   ----- ---------------------------------- 1/8 [Werkzeug]
   ----- ---------------------------------- 1/8 [Werkzeug]
   ----- ---------------------------------- 1/8 [Werkzeug]
   ----- ---------------------------------- 1/8 [Werkzeug]
   ---------- ----------------------------- 2/8 [retrying]
   -------------------- ------------------- 4/8 [blinker]
   ------------------------- -


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


###### Save App to File

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import warnings
warnings.filterwarnings('ignore')
 
# ── 1. Load Cleaned Data ──────────────────────────────────────────────────────
df = pd.read_csv('C:\\Users\\User\\Desktop\\DV\\DV_project\\Melbourne_Housing_Cleaned.csv')
df['Date']         = pd.to_datetime(df['Date'])
df['YearMonth']    = df['Date'].dt.to_period('M').astype(str)
type_map           = {'h': 'House', 't': 'Townhouse', 'u': 'Unit'}
df['PropertyType'] = df['Type'].map(type_map)
df['Price_M']      = df['Price'] / 1e6
 
COLORS = {
    'House':     '#2E86AB',
    'Townhouse': '#A23B72',
    'Unit':      '#F18F01',
    'All':       '#3B1F2B'
}
BG = '#F8F9FA'
 
# ── 2. App Layout ─────────────────────────────────────────────────────────────
app = Dash(__name__)
app.title = "Melbourne Housing Dashboard"
 
app.layout = html.Div(
    style={'backgroundColor': BG, 'fontFamily': 'Arial, sans-serif',
           'minHeight': '100vh', 'padding': '0 0 40px 0'},
    children=[
 
        # Header
        html.Div(
            style={'backgroundColor': '#2E86AB', 'padding': '24px 40px',
                   'marginBottom': '30px'},
            children=[
                html.H1("Melbourne Housing Market Dashboard",
                        style={'color': 'white', 'margin': 0,
                               'fontSize': '26px', 'fontWeight': 'bold'}),
                html.P("Interactive exploration of the Melbourne Housing Dataset (2016-2017)",
                       style={'color': '#d0eaf7', 'margin': '6px 0 0 0',
                              'fontSize': '14px'})
            ]
        ),
 
        # Controls
        html.Div(
            style={'display': 'flex', 'gap': '40px', 'alignItems': 'flex-end',
                   'padding': '0 40px', 'marginBottom': '30px', 'flexWrap': 'wrap'},
            children=[
                html.Div([
                    html.Label("Property Type",
                               style={'fontWeight': 'bold', 'fontSize': '13px',
                                      'color': '#333', 'marginBottom': '6px',
                                      'display': 'block'}),
                    dcc.Dropdown(
                        id='type-dropdown',
                        options=[{'label': 'All Types', 'value': 'All'}] +
                                [{'label': t, 'value': t}
                                 for t in ['House', 'Townhouse', 'Unit']],
                        value='All', clearable=False,
                        style={'width': '200px', 'fontSize': '13px'}
                    )
                ]),
                html.Div([
                    html.Label(id='slider-label',
                               style={'fontWeight': 'bold', 'fontSize': '13px',
                                      'color': '#333', 'marginBottom': '6px',
                                      'display': 'block'}),
                    dcc.Slider(
                        id='price-slider', min=0, max=5, step=0.5, value=5,
                        marks={i: f'${i}M' for i in range(0, 6)},
                        tooltip={'placement': 'bottom', 'always_visible': False}
                    )
                ], style={'width': '340px'}),
                html.Div([
                    html.Label("Scatter Opacity",
                               style={'fontWeight': 'bold', 'fontSize': '13px',
                                      'color': '#333', 'marginBottom': '6px',
                                      'display': 'block'}),
                    dcc.RadioItems(
                        id='opacity-radio',
                        options=[
                            {'label': ' Light',  'value': 0.3},
                            {'label': ' Medium', 'value': 0.6},
                            {'label': ' Full',   'value': 1.0},
                        ],
                        value=0.6, inline=True,
                        style={'fontSize': '13px', 'gap': '16px'}
                    )
                ]),
            ]
        ),
 
        # Charts Row
        html.Div(
            style={'display': 'flex', 'gap': '24px', 'padding': '0 40px',
                   'flexWrap': 'wrap'},
            children=[
                html.Div(
                    style={'flex': '1', 'minWidth': '420px',
                           'backgroundColor': 'white', 'borderRadius': '10px',
                           'padding': '20px', 'boxShadow': '0 2px 8px rgba(0,0,0,0.08)'},
                    children=[
                        html.H3("Building Area vs. Sale Price",
                                style={'margin': '0 0 4px 0', 'fontSize': '15px',
                                       'color': '#222', 'fontWeight': 'bold'}),
                        html.P("Filtered by property type and max price",
                               style={'margin': '0 0 12px 0', 'fontSize': '12px',
                                      'color': '#888'}),
                        dcc.Graph(id='scatter-chart', config={'displayModeBar': False})
                    ]
                ),
                html.Div(
                    style={'flex': '1', 'minWidth': '420px',
                           'backgroundColor': 'white', 'borderRadius': '10px',
                           'padding': '20px', 'boxShadow': '0 2px 8px rgba(0,0,0,0.08)'},
                    children=[
                        html.H3("Average Price Over Time",
                                style={'margin': '0 0 4px 0', 'fontSize': '15px',
                                       'color': '#222', 'fontWeight': 'bold'}),
                        html.P("Monthly average sale price by property type",
                               style={'margin': '0 0 12px 0', 'fontSize': '12px',
                                      'color': '#888'}),
                        dcc.Graph(id='line-chart', config={'displayModeBar': False})
                    ]
                ),
            ]
        ),
 
        # KPI Cards
        html.Div(id='kpi-row',
                 style={'display': 'flex', 'gap': '20px',
                        'padding': '24px 40px 0 40px', 'flexWrap': 'wrap'}),
    ]
)
 
 
# ── 3. Callbacks ──────────────────────────────────────────────────────────────
 
@app.callback(
    Output('slider-label', 'children'),
    Input('price-slider', 'value')
)
def update_slider_label(max_price):
    return f"Max Price: ${max_price}M"
 
 
@app.callback(
    Output('scatter-chart', 'figure'),
    Input('type-dropdown', 'value'),
    Input('price-slider', 'value'),
    Input('opacity-radio', 'value')
)
def update_scatter(selected_type, max_price, opacity):
    dff = df[df['Price_M'] <= max_price].copy()
    dff = dff[dff['BuildingArea'] < 600]
    if selected_type != 'All':
        dff = dff[dff['PropertyType'] == selected_type]
 
    fig = px.scatter(
        dff, x='BuildingArea', y='Price_M',
        color='PropertyType', color_discrete_map=COLORS,
        opacity=opacity, trendline='ols', trendline_scope='overall',
        labels={'BuildingArea': 'Building Area (m2)',
                'Price_M': 'Sale Price (AUD Millions)',
                'PropertyType': 'Type'},
        hover_data={'Suburb': True, 'Rooms': True,
                    'Price_M': ':.2f', 'BuildingArea': True}
    )
    fig.update_layout(
        plot_bgcolor=BG, paper_bgcolor='white',
        margin=dict(l=10, r=10, t=10, b=10),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, title=''),
        font=dict(family='Arial', size=12), height=340
    )
    fig.update_xaxes(gridcolor='#ECECEC', title_font_size=12)
    fig.update_yaxes(gridcolor='#ECECEC', tickprefix='$', ticksuffix='M',
                     title_font_size=12)
    return fig
 
 
@app.callback(
    Output('line-chart', 'figure'),
    Input('type-dropdown', 'value'),
    Input('price-slider', 'value')
)
def update_line(selected_type, max_price):
    dff = df[df['Price_M'] <= max_price].copy()
    types_to_show = ([selected_type] if selected_type != 'All'
                     else ['House', 'Townhouse', 'Unit'])
 
    fig = go.Figure()
    for ptype in types_to_show:
        sub = dff[dff['PropertyType'] == ptype]
        line_data = (sub.groupby('YearMonth')['Price_M']
                       .mean().reset_index()
                       .sort_values('YearMonth'))
        fig.add_trace(go.Scatter(
            x=line_data['YearMonth'], y=line_data['Price_M'],
            mode='lines+markers', name=ptype,
            line=dict(color=COLORS[ptype], width=2.5),
            marker=dict(size=5),
            hovertemplate=(
                f"<b>{ptype}</b><br>"
                "Month: %{x}<br>"
                "Avg Price: $%{y:.2f}M<extra></extra>"
            )
        ))
 
    fig.update_layout(
        plot_bgcolor=BG, paper_bgcolor='white',
        margin=dict(l=10, r=10, t=10, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, title=''),
        font=dict(family='Arial', size=12), height=340,
        xaxis=dict(tickangle=45, gridcolor='#ECECEC', title='Sale Month'),
        yaxis=dict(gridcolor='#ECECEC', tickprefix='$', ticksuffix='M',
                   title='Avg Price (AUD M)')
    )
    return fig
 
 
@app.callback(
    Output('kpi-row', 'children'),
    Input('type-dropdown', 'value'),
    Input('price-slider', 'value')
)
def update_kpis(selected_type, max_price):
    dff = df[df['Price_M'] <= max_price].copy()
    if selected_type != 'All':
        dff = dff[dff['PropertyType'] == selected_type]
 
    kpis = [
        ("Total Properties", f"{len(dff):,}",                    "#2E86AB"),
        ("Median Price",     f"${dff['Price_M'].median():.2f}M", "#A23B72"),
        ("Average Price",    f"${dff['Price_M'].mean():.2f}M",   "#F18F01"),
        ("Avg Rooms",        f"{dff['Rooms'].mean():.1f}",        "#3B1F2B"),
    ]
 
    cards = []
    for label, value, color in kpis:
        cards.append(html.Div(
            style={'flex': '1', 'minWidth': '140px',
                   'backgroundColor': 'white', 'borderRadius': '10px',
                   'padding': '16px 20px',
                   'borderLeft': f'5px solid {color}',
                   'boxShadow': '0 2px 8px rgba(0,0,0,0.07)'},
            children=[
                html.P(label, style={'margin': '0 0 4px 0', 'fontSize': '12px',
                                     'color': '#888', 'fontWeight': 'bold',
                                     'textTransform': 'uppercase',
                                     'letterSpacing': '0.5px'}),
                html.H2(value, style={'margin': 0, 'fontSize': '22px',
                                      'color': color, 'fontWeight': 'bold'})
            ]
        ))
    return cards
 
 
# ── 4. Run ────────────────────────────────────────────────────────────────────
server = app.server

if __name__ == '__main__':
    app.run(debug=True)